# Real data analysis project in `duckdb`

This is again just the prologue to allow the notebooks running standalone in a local Jupyter installation and in Google Colab.

In [1]:
import os
try:
    import google.colab
    IN_COLAB = True
    if not os.path.isdir("/content/oreilly-duckdb"):
        os.system("git clone https://github.com/datanizing/oreilly-duckdb")
    data = "/content/oreilly-duckdb/data"
except:
    IN_COLAB = False
    data = "data"

## Download data

In [2]:
from tqdm.auto import tqdm
import requests

response = requests.get("https://www.kaggle.com/api/v1/datasets/download/Cornell-University/arxiv", stream=True)
total_size = int(response.headers.get("content-length", 0))
block_size = 1024

with tqdm(total=total_size, unit="B", unit_scale=True) as progress_bar:
    with open("arxiv.zip", "wb") as file:
        for data in response.iter_content(block_size):
            progress_bar.update(len(data))
            file.write(data)

if total_size != 0 and progress_bar.n != total_size:
    print("Could not download file")

  0%|          | 0.00/1.80G [00:00<?, ?B/s]

In [4]:
import duckdb
duckdb.sql("INSTALL zipfs FROM community; LOAD zipfs") 

If the extension does not work for you, you can 
extract he files manually or use the following code
to extract the files (adjust the names in the `duckdb`
statements appropriately):
```python
from zipfile import ZipFile
with ZipFile("arxiv.zip", 'r') as zip_ref:
    zip_ref.extractall(".")
```

First convert to `parquet` format so we have fast access. We will use
the current timestamp in the converted `parquet` file. There are a 
few reasons for that:
* We don't want to overwrite an already existing converted `parquet` file.
* If we use more sophisticated analysis (like NLP or LLMs) on the data, we
  would like to restrict that on the new data (incremental updates). 

In [5]:
from datetime import datetime
today = datetime.now().strftime('%Y%m%d')

In [7]:
%%time
duckdb.sql(f"COPY (SELECT * FROM 'zip://arxiv.zip/arxiv-metadata-oai-snapshot.json') TO 'arxiv-{today}.parquet'")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 1min 37s, sys: 13.2 s, total: 1min 50s
Wall time: 41.4 s


In [9]:
duckdb.sql(f"FROM 'arxiv-{today}.parquet' LIMIT 5").pl()

id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
str,str,str,str,str,str,str,str,str,str,str,list[struct[2]],date,list[list[str]]
"""0704.0001""","""Pavel Nadolsky""","""C. Bal\'azs, E. L. Berger, P. …","""Calculation of prompt diphoton…","""37 pages, 15 figures; publishe…","""Phys.Rev.D76:013009,2007""","""10.1103/PhysRevD.76.013009""","""ANL-HEP-PR-07-12""","""hep-ph""",null,""" A fully differential calcula…","[{""v1"",""Mon, 2 Apr 2007 19:18:42 GMT""}, {""v2"",""Tue, 24 Jul 2007 20:10:27 GMT""}]",2008-11-26,"[[""Balázs"", ""C."", """"], [""Berger"", ""E. L."", """"], … [""Yuan"", ""C. -P."", """"]]"
"""0704.0002""","""Louis Theran""","""Ileana Streinu and Louis Thera…","""Sparsity-certifying Graph Deco…","""To appear in Graphs and Combin…",null,null,null,"""math.CO cs.CG""","""http://arxiv.org/licenses/none…",""" We describe a new algorithm,…","[{""v1"",""Sat, 31 Mar 2007 02:26:18 GMT""}, {""v2"",""Sat, 13 Dec 2008 17:26:00 GMT""}]",2008-12-13,"[[""Streinu"", ""Ileana"", """"], [""Theran"", ""Louis"", """"]]"
"""0704.0003""","""Hongjun Pan""","""Hongjun Pan""","""The evolution of the Earth-Moo…","""23 pages, 3 figures""",null,null,null,"""physics.gen-ph""",null,""" The evolution of Earth-Moon …","[{""v1"",""Sun, 1 Apr 2007 20:46:54 GMT""}, {""v2"",""Sat, 8 Dec 2007 23:47:24 GMT""}, {""v3"",""Sun, 13 Jan 2008 00:36:28 GMT""}]",2008-01-13,"[[""Pan"", ""Hongjun"", """"]]"
"""0704.0004""","""David Callan""","""David Callan""","""A determinant of Stirling cycl…","""11 pages""",null,null,null,"""math.CO""",null,""" We show that a determinant o…","[{""v1"",""Sat, 31 Mar 2007 03:16:14 GMT""}]",2007-05-23,"[[""Callan"", ""David"", """"]]"
"""0704.0005""","""Alberto Torchinsky""","""Wael Abu-Shammala and Alberto …","""From dyadic $\Lambda_{\alpha}$…",null,"""Illinois J. Math. 52 (2008) no…",null,null,"""math.CA math.FA""",null,""" In this paper we show how to…","[{""v1"",""Mon, 2 Apr 2007 18:09:58 GMT""}]",2013-10-15,"[[""Abu-Shammala"", ""Wael"", """"], [""Torchinsky"", ""Alberto"", """"]]"


In [40]:
duckdb.sql(f"SELECT count(*) FROM 'arxiv-{today}.parquet' LIMIT 5").pl()

count_star()
i64
3120363


In [19]:
duckdb.sql(f"DESCRIBE FROM 'arxiv-{today}.parquet'").pl().show(limit=15, fmt_str_lengths=100)

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""id""","""VARCHAR""","""YES""",null,null,null
"""submitter""","""VARCHAR""","""YES""",null,null,null
"""authors""","""VARCHAR""","""YES""",null,null,null
"""title""","""VARCHAR""","""YES""",null,null,null
"""comments""","""VARCHAR""","""YES""",null,null,null
"""journal-ref""","""VARCHAR""","""YES""",null,null,null
"""doi""","""VARCHAR""","""YES""",null,null,null
"""report-no""","""VARCHAR""","""YES""",null,null,null
"""categories""","""VARCHAR""","""YES""",null,null,null


In [29]:
duckdb.sql(f"SELECT id, unnest(versions) FROM 'arxiv-{today}.parquet' LIMIT 5").pl()

id,unnest(versions)
str,struct[2]
"""0704.0001""","{""v1"",""Mon, 2 Apr 2007 19:18:42 GMT""}"
"""0704.0001""","{""v2"",""Tue, 24 Jul 2007 20:10:27 GMT""}"
"""0704.0002""","{""v1"",""Sat, 31 Mar 2007 02:26:18 GMT""}"
"""0704.0002""","{""v2"",""Sat, 13 Dec 2008 17:26:00 GMT""}"
"""0704.0003""","{""v1"",""Sun, 1 Apr 2007 20:46:54 GMT""}"


In [28]:
duckdb.sql(f"SELECT id, unnest(versions, recursive := true) FROM 'arxiv-{today}.parquet' LIMIT 5").pl()

id,version,created
str,str,str
"""0704.0001""","""v1""","""Mon, 2 Apr 2007 19:18:42 GMT"""
"""0704.0001""","""v2""","""Tue, 24 Jul 2007 20:10:27 GMT"""
"""0704.0002""","""v1""","""Sat, 31 Mar 2007 02:26:18 GMT"""
"""0704.0002""","""v2""","""Sat, 13 Dec 2008 17:26:00 GMT"""
"""0704.0003""","""v1""","""Sun, 1 Apr 2007 20:46:54 GMT"""


In [35]:
duckdb.sql(f"SELECT id, unnest(versions, recursive := true) FROM 'arxiv-{today}.parquet' WHERE version='v1' LIMIT 5").pl()

BinderException: Binder Error: Referenced column "version" not found in FROM clause!
Candidate bindings: "versions", "report-no", "categories", "comments", "license"

In [41]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT * EXCLUDE(versions) FROM publications WHERE version='v1' LIMIT 5").pl()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,update_date,authors_parsed,version,created
str,str,str,str,str,str,str,str,str,str,str,date,list[list[str]],str,str
"""0704.0001""","""Pavel Nadolsky""","""C. Bal\'azs, E. L. Berger, P. …","""Calculation of prompt diphoton…","""37 pages, 15 figures; publishe…","""Phys.Rev.D76:013009,2007""","""10.1103/PhysRevD.76.013009""","""ANL-HEP-PR-07-12""","""hep-ph""",null,""" A fully differential calcula…",2008-11-26,"[[""Balázs"", ""C."", """"], [""Berger"", ""E. L."", """"], … [""Yuan"", ""C. -P."", """"]]","""v1""","""Mon, 2 Apr 2007 19:18:42 GMT"""
"""0704.0002""","""Louis Theran""","""Ileana Streinu and Louis Thera…","""Sparsity-certifying Graph Deco…","""To appear in Graphs and Combin…",null,null,null,"""math.CO cs.CG""","""http://arxiv.org/licenses/none…",""" We describe a new algorithm,…",2008-12-13,"[[""Streinu"", ""Ileana"", """"], [""Theran"", ""Louis"", """"]]","""v1""","""Sat, 31 Mar 2007 02:26:18 GMT"""
"""0704.0003""","""Hongjun Pan""","""Hongjun Pan""","""The evolution of the Earth-Moo…","""23 pages, 3 figures""",null,null,null,"""physics.gen-ph""",null,""" The evolution of Earth-Moon …",2008-01-13,"[[""Pan"", ""Hongjun"", """"]]","""v1""","""Sun, 1 Apr 2007 20:46:54 GMT"""
"""0704.0004""","""David Callan""","""David Callan""","""A determinant of Stirling cycl…","""11 pages""",null,null,null,"""math.CO""",null,""" We show that a determinant o…",2007-05-23,"[[""Callan"", ""David"", """"]]","""v1""","""Sat, 31 Mar 2007 03:16:14 GMT"""
"""0704.0005""","""Alberto Torchinsky""","""Wael Abu-Shammala and Alberto …","""From dyadic $\Lambda_{\alpha}$…",null,"""Illinois J. Math. 52 (2008) no…",null,null,"""math.CA math.FA""",null,""" In this paper we show how to…",2013-10-15,"[[""Abu-Shammala"", ""Wael"", """"], [""Torchinsky"", ""Alberto"", """"]]","""v1""","""Mon, 2 Apr 2007 18:09:58 GMT"""


Why so slow? All columns must be read. For now only `id` and `created`:

In [43]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT id, created FROM publications WHERE version='v1' LIMIT 5").pl()

id,created
str,str
"""0704.0001""","""Mon, 2 Apr 2007 19:18:42 GMT"""
"""0704.0002""","""Sat, 31 Mar 2007 02:26:18 GMT"""
"""0704.0003""","""Sun, 1 Apr 2007 20:46:54 GMT"""
"""0704.0004""","""Sat, 31 Mar 2007 03:16:14 GMT"""
"""0704.0005""","""Mon, 2 Apr 2007 18:09:58 GMT"""


Now parse the date

In [47]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT id, created, strptime(created, '%a, %d %b %Y %H:%M:%S %Z') FROM publications WHERE version='v1' LIMIT 5").pl()

id,created,"strptime(created, '%a, %d %b %Y %H:%M:%S %Z')"
str,str,"datetime[μs, Etc/UTC]"
"""0704.0001""","""Mon, 2 Apr 2007 19:18:42 GMT""",2007-04-02 19:18:42 UTC
"""0704.0002""","""Sat, 31 Mar 2007 02:26:18 GMT""",2007-03-31 02:26:18 UTC
"""0704.0003""","""Sun, 1 Apr 2007 20:46:54 GMT""",2007-04-01 20:46:54 UTC
"""0704.0004""","""Sat, 31 Mar 2007 03:16:14 GMT""",2007-03-31 03:16:14 UTC
"""0704.0005""","""Mon, 2 Apr 2007 18:09:58 GMT""",2007-04-02 18:09:58 UTC


Make categories a list

In [49]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT id, string_split(categories, ' ') FROM publications WHERE version='v1' LIMIT 5").pl()

id,"string_split(categories, ' ')"
str,list[str]
"""0704.0001""","[""hep-ph""]"
"""0704.0002""","[""math.CO"", ""cs.CG""]"
"""0704.0003""","[""physics.gen-ph""]"
"""0704.0004""","[""math.CO""]"
"""0704.0005""","[""math.CA"", ""math.FA""]"


Save intermediate results as `parquet`

In [53]:
duckdb.sql(f"DESCRIBE FROM 'arxiv-{today}.parquet'").pl().show(limit=15, fmt_str_lengths=100)

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""id""","""VARCHAR""","""YES""",null,null,null
"""submitter""","""VARCHAR""","""YES""",null,null,null
"""authors""","""VARCHAR""","""YES""",null,null,null
"""title""","""VARCHAR""","""YES""",null,null,null
"""comments""","""VARCHAR""","""YES""",null,null,null
"""journal-ref""","""VARCHAR""","""YES""",null,null,null
"""doi""","""VARCHAR""","""YES""",null,null,null
"""report-no""","""VARCHAR""","""YES""",null,null,null
"""categories""","""VARCHAR""","""YES""",null,null,null


In [54]:
duckdb.sql(f"WITH publications AS (SELECT *, unnest(versions, recursive := true), FROM 'arxiv-{today}.parquet')\
             SELECT *, string_split(categories, ' ') AS categories, FROM publications WHERE version='v1' LIMIT 5").pl()

id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed,version,created,categories_1
str,str,str,str,str,str,str,str,str,str,str,list[struct[2]],date,list[list[str]],str,str,list[str]
"""0704.0001""","""Pavel Nadolsky""","""C. Bal\'azs, E. L. Berger, P. …","""Calculation of prompt diphoton…","""37 pages, 15 figures; publishe…","""Phys.Rev.D76:013009,2007""","""10.1103/PhysRevD.76.013009""","""ANL-HEP-PR-07-12""","""hep-ph""",null,""" A fully differential calcula…","[{""v1"",""Mon, 2 Apr 2007 19:18:42 GMT""}, {""v2"",""Tue, 24 Jul 2007 20:10:27 GMT""}]",2008-11-26,"[[""Balázs"", ""C."", """"], [""Berger"", ""E. L."", """"], … [""Yuan"", ""C. -P."", """"]]","""v1""","""Mon, 2 Apr 2007 19:18:42 GMT""","[""hep-ph""]"
"""0704.0002""","""Louis Theran""","""Ileana Streinu and Louis Thera…","""Sparsity-certifying Graph Deco…","""To appear in Graphs and Combin…",null,null,null,"""math.CO cs.CG""","""http://arxiv.org/licenses/none…",""" We describe a new algorithm,…","[{""v1"",""Sat, 31 Mar 2007 02:26:18 GMT""}, {""v2"",""Sat, 13 Dec 2008 17:26:00 GMT""}]",2008-12-13,"[[""Streinu"", ""Ileana"", """"], [""Theran"", ""Louis"", """"]]","""v1""","""Sat, 31 Mar 2007 02:26:18 GMT""","[""math.CO"", ""cs.CG""]"
"""0704.0003""","""Hongjun Pan""","""Hongjun Pan""","""The evolution of the Earth-Moo…","""23 pages, 3 figures""",null,null,null,"""physics.gen-ph""",null,""" The evolution of Earth-Moon …","[{""v1"",""Sun, 1 Apr 2007 20:46:54 GMT""}, {""v2"",""Sat, 8 Dec 2007 23:47:24 GMT""}, {""v3"",""Sun, 13 Jan 2008 00:36:28 GMT""}]",2008-01-13,"[[""Pan"", ""Hongjun"", """"]]","""v1""","""Sun, 1 Apr 2007 20:46:54 GMT""","[""physics.gen-ph""]"
"""0704.0004""","""David Callan""","""David Callan""","""A determinant of Stirling cycl…","""11 pages""",null,null,null,"""math.CO""",null,""" We show that a determinant o…","[{""v1"",""Sat, 31 Mar 2007 03:16:14 GMT""}]",2007-05-23,"[[""Callan"", ""David"", """"]]","""v1""","""Sat, 31 Mar 2007 03:16:14 GMT""","[""math.CO""]"
"""0704.0005""","""Alberto Torchinsky""","""Wael Abu-Shammala and Alberto …","""From dyadic $\Lambda_{\alpha}$…",null,"""Illinois J. Math. 52 (2008) no…",null,null,"""math.CA math.FA""",null,""" In this paper we show how to…","[{""v1"",""Mon, 2 Apr 2007 18:09:58 GMT""}]",2013-10-15,"[[""Abu-Shammala"", ""Wael"", """"], [""Torchinsky"", ""Alberto"", """"]]","""v1""","""Mon, 2 Apr 2007 18:09:58 GMT""","[""math.CA"", ""math.FA""]"


Find duplicates

Most common authors

Heatmap

Time development

Forecast